In [3]:
!pip install transformers datasets rouge-score sentencepiece -q

In [1]:
from datasets import load_dataset

dataset = load_dataset("abisee/cnn_dailymail", "3.0.0")

train_data = dataset["train"].select(range(1000))
test_data = dataset["test"].select(range(50))

In [2]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/flan-t5-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


In [4]:
def preprocess(examples):
    inputs = ["summarize: " + doc for doc in examples["article"]]
    model_inputs = tokenizer(inputs, max_length=512, truncation=True, padding="max_length")
    labels = tokenizer(examples["highlights"], max_length=128, truncation=True, padding="max_length")
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_train = train_data.map(preprocess, batched=True)
tokenized_test = test_data.map(preprocess, batched=True)

In [ ]:
dont run cell 5 (baseline)

In [5]:
import torch
from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

def evaluate(data, n=25):
    r1, r2, rl = [], [], []
    model.eval()
    for i in range(n):
        article = data[i]["article"][:1024]
        reference = data[i]["highlights"]
        inputs = tokenizer("summarize: " + article, return_tensors="pt", max_length=512, truncation=True).to(device)
        with torch.no_grad():
            outputs = model.generate(**inputs, max_new_tokens=128)
        generated = tokenizer.decode(outputs[0], skip_special_tokens=True)
        scores = scorer.score(reference, generated)
        r1.append(scores["rouge1"].fmeasure)
        r2.append(scores["rouge2"].fmeasure)
        rl.append(scores["rougeL"].fmeasure)
    print(f"ROUGE-1: {sum(r1)/len(r1):.4f}")
    print(f"ROUGE-2: {sum(r2)/len(r2):.4f}")
    print(f"ROUGE-L: {sum(rl)/len(rl):.4f}")

print("BEFORE training:")
evaluate(test_data)

BEFORE training:
ROUGE-1: 0.1795
ROUGE-2: 0.0548
ROUGE-L: 0.1363


In [13]:
!pip install accelerate -q

In [6]:
from transformers import TrainingArguments, Trainer

args = TrainingArguments(
    output_dir="./flan-t5-finetuned",
    num_train_epochs=1,
    per_device_train_batch_size=4,
    save_strategy="epoch",
    logging_steps=50,
    fp16=True,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_train,
)

trainer.train()

Step,Training Loss
50,0.000000
100,0.000000
150,0.000000
200,0.000000
250,0.000000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=250, training_loss=0.0, metrics={'train_runtime': 36.8321, 'train_samples_per_second': 27.15, 'train_steps_per_second': 6.788, 'total_flos': 185890504704000.0, 'train_loss': 0.0, 'epoch': 1.0})

In [8]:
from transformers import DataCollatorForSeq2Seq
from transformers import TrainingArguments, Trainer

def preprocess_fixed(examples):
    inputs = ["summarize: " + doc for doc in examples["article"]]
    targets = examples["highlights"]
    model_inputs = tokenizer(inputs, max_length=512, truncation=True, padding=False)
    labels = tokenizer(text_target=targets, max_length=128, truncation=True, padding=False)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_train_fixed = train_data.map(preprocess_fixed, batched=True, remove_columns=train_data.column_names)
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model, padding=True)

args = TrainingArguments(
    output_dir="./flan-t5-finetuned",
    num_train_epochs=1,
    per_device_train_batch_size=4,
    save_strategy="epoch",
    logging_steps=50,
    fp16=False,
    bf16=True,
    learning_rate=5e-4,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_train_fixed,
    data_collator=data_collator,
)

trainer.train()

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Step,Training Loss
50,0.000000
100,0.000000
150,0.000000
200,0.000000
250,0.000000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=250, training_loss=0.0, metrics={'train_runtime': 39.2456, 'train_samples_per_second': 25.481, 'train_steps_per_second': 6.37, 'total_flos': 185890504704000.0, 'train_loss': 0.0, 'epoch': 1.0})

In [9]:
sample = tokenized_train_fixed[0]
print("Input length:", len(sample["input_ids"]))
print("Label length:", len(sample["labels"]))
print("First 20 input tokens:", sample["input_ids"][:20])
print("First 20 label tokens:", sample["labels"][:20])
print("Label decoded:", tokenizer.decode(sample["labels"], skip_special_tokens=True))


Input length: 512
Label length: 57
First 20 input tokens: [21603, 10, 301, 24796, 4170, 6, 2789, 41, 18844, 61, 1636, 8929, 16023, 2213, 4173, 6324, 12591, 15, 11391, 592]
First 20 label tokens: [8929, 16023, 2213, 4173, 6324, 12591, 15, 2347, 3996, 1755, 329, 13462, 38, 3, 88, 5050, 507, 2089, 3, 5]
Label decoded: Harry Potter star Daniel Radcliffe gets £20M fortune as he turns 18 Monday . Young actor says he has no plans to fritter his cash away . Radcliffe's earnings from first five Potter films have been held in trust fund .


In [10]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, TrainingArguments, Trainer, DataCollatorForSeq2Seq

# Reload fresh model
tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-small")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small")

# Retokenize
def preprocess_fixed(examples):
    inputs = ["summarize: " + doc for doc in examples["article"]]
    targets = examples["highlights"]
    model_inputs = tokenizer(inputs, max_length=512, truncation=True, padding=False)
    labels = tokenizer(text_target=targets, max_length=128, truncation=True, padding=False)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_train_fixed = train_data.map(preprocess_fixed, batched=True, remove_columns=train_data.column_names)
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model, padding=True)

args = TrainingArguments(
    output_dir="./flan-t5-finetuned",
    num_train_epochs=1,
    per_device_train_batch_size=4,
    save_strategy="epoch",
    logging_steps=50,
    learning_rate=5e-4,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_train_fixed,
    data_collator=data_collator,
)

trainer.train()

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Step,Training Loss
50,2.486556
100,2.413246
150,2.418555
200,2.395574
250,2.403454


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=250, training_loss=2.42347705078125, metrics={'train_runtime': 35.1451, 'train_samples_per_second': 28.453, 'train_steps_per_second': 7.113, 'total_flos': 185890504704000.0, 'train_loss': 2.42347705078125, 'epoch': 1.0})

In [11]:
import torch
from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

def evaluate(data, n=25):
    r1, r2, rl = [], [], []
    model.eval()
    for i in range(n):
        article = data[i]["article"][:1024]
        reference = data[i]["highlights"]
        inputs = tokenizer("summarize: " + article, return_tensors="pt", max_length=512, truncation=True).to(device)
        with torch.no_grad():
            outputs = model.generate(**inputs, max_new_tokens=128)
        generated = tokenizer.decode(outputs[0], skip_special_tokens=True)
        scores = scorer.score(reference, generated)
        r1.append(scores["rouge1"].fmeasure)
        r2.append(scores["rouge2"].fmeasure)
        rl.append(scores["rougeL"].fmeasure)
    print(f"ROUGE-1: {sum(r1)/len(r1):.4f}")
    print(f"ROUGE-2: {sum(r2)/len(r2):.4f}")
    print(f"ROUGE-L: {sum(rl)/len(rl):.4f}")

print("AFTER training:")
evaluate(test_data)

# Save the fine-tuned model
model.save_pretrained("./flan-t5-finetuned")
tokenizer.save_pretrained("./flan-t5-finetuned")
print("\nModel saved to ./flan-t5-finetuned")

AFTER training:
ROUGE-1: 0.2743
ROUGE-2: 0.0956
ROUGE-L: 0.2080


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Model saved to ./flan-t5-finetuned


In [12]:
import os
print(os.getcwd())

C:\Users\julia
